# Core-Cusp Analyse im UQSH-Rahmen

**Autor:** Ukshin Q. Rexhepi  
**Zweck:** Vollständige Analyse der inneren Massenprofile aus SPARC-Rotationskurven  
**Ergebnisse:** Drei empirisch unterscheidbare Regime der Core-Cusp Dynamik

## Struktur
1. Setup und Datenpfade
2. Daten laden (SPARC ROTMOD + q-Fits)
3. Innere Steigungen berechnen
4. Größenabhängigkeit: drei Regime
5. Gamma-Sättigungsterm (Klein vs. Groß)
6. Abbildungen für Paper
7. Ergebnisse speichern

## 0. Setup

In [ ]:
# Google Drive mounten (nur in Colab nötig)
# from google.colab import drive
# drive.mount('/content/drive')

# Pfade anpassen falls nötig
# ZIP_PATH = '/content/drive/MyDrive/UQSH/Rotmod_LTG.zip'
# Q_PATH   = '/content/drive/MyDrive/UQSH/q_fit_results.csv'
# OUT_PATH = '/content/drive/MyDrive/UQSH/results/'

# Standard: direkt hochgeladen in Colab
ZIP_PATH = '/content/Rotmod_LTG.zip'
Q_PATH   = '/content/q_fit_results.csv'
OUT_PATH = '/content/'

In [ ]:
import os
import glob
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

EXTRACT_PATH = '/content/rotmod'
SCHWELLE_KLEIN     = 16.0   # kpc: Klein vs. Übergang
SCHWELLE_UEBERGANG = 30.0   # kpc: Übergang vs. Groß

# Gamma-Term Konfigurationen
GAMMA_CONFIGS = [
    {'label': 'g_schwach',    'A_gamma': 0.05, 'r_gamma': 1.0, 'phase': False},
    {'label': 'g_mittel',     'A_gamma': 0.15, 'r_gamma': 2.0, 'phase': False},
    {'label': 'g_stark',      'A_gamma': 0.30, 'r_gamma': 3.0, 'phase': False},
    {'label': 'g_phase_halb', 'A_gamma': 0.15, 'r_gamma': 2.0, 'phase': True, 'f_lambda': 0.5},
    {'label': 'g_phase_ganz', 'A_gamma': 0.15, 'r_gamma': 2.0, 'phase': True, 'f_lambda': 1.0},
]

COLORS = {'Peak': 'steelblue', 'Transition': 'orange', 'Diffus': 'green'}

print('Setup abgeschlossen.')

## 1. Daten laden

In [ ]:
# ZIP entpacken
if not os.path.exists(EXTRACT_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)
    print(f'Entpackt nach {EXTRACT_PATH}')
else:
    print('Bereits entpackt.')

files = glob.glob(EXTRACT_PATH + '/**/*.dat', recursive=True)
print(f'Gefundene .dat Dateien: {len(files)}')

In [ ]:
# ROTMOD Dateien laden
all_data = []

for f in files:
    try:
        df = pd.read_csv(f, sep=r'\s+', header=None,
                         comment='#', engine='python')
        if df.shape[1] < 6:
            continue
        df = df.iloc[:, :8].copy()
        while df.shape[1] < 8:
            df[df.shape[1]] = pd.NA
        df.columns = ['r','Vobs','errV','Vgas',
                      'Vdisk','Vbul','col7','col8']
        df['galaxy'] = Path(f).stem.replace('_rotmod','')
        for col in ['r','Vobs','errV','Vgas','Vdisk','Vbul']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.dropna(subset=['r','Vobs','Vgas','Vdisk','Vbul'])
        df = df[df['r'] > 0]
        if len(df) > 0:
            all_data.append(df)
    except Exception:
        continue

df_all = pd.concat(all_data, ignore_index=True)

# Basisgrößen
df_all['M_eff'] = df_all['r'] * df_all['Vobs']**2
df_all['M_bar'] = df_all['r'] * (df_all['Vgas']**2
                                 + df_all['Vdisk']**2
                                 + df_all['Vbul']**2)
df_all['g_obs'] = df_all['Vobs']**2 / df_all['r']
df_all['g_bar'] = df_all['M_bar'] / df_all['r']**2

print(f'Datenpunkte: {len(df_all)}')
print(f'Galaxien:    {df_all["galaxy"].nunique()}')

In [ ]:
# q-Fit Ergebnisse laden und Regime klassifizieren
q_df = pd.read_csv(Q_PATH)
if 'filename' in q_df.columns:
    q_df['galaxy'] = q_df['filename'].str.replace(
        '_rotmod.dat', '', regex=False)

def classify_q(q):
    if pd.isna(q): return np.nan
    if q < 1.2:    return 'Peak'
    if q > 2.5:    return 'Diffus'
    return 'Transition'

q_df['regime_q'] = q_df['q'].apply(classify_q)
print(q_df['regime_q'].value_counts())

## 2. Innere Steigungen und Galaxy-Eigenschaften

In [ ]:
def inner_slope_from_profile(r_vals, M_vals, n_inner=5):
    """Logarithmische Steigung dlogM/dlogr aus n_inner innersten Punkten."""
    idx  = np.argsort(r_vals)
    r_s  = r_vals[idx][:n_inner]
    M_s  = M_vals[idx][:n_inner]
    mask = (r_s > 0) & (M_s > 0)
    if mask.sum() < 3:
        return np.nan
    return np.polyfit(np.log10(r_s[mask]),
                      np.log10(M_s[mask]), 1)[0]

def gamma_term(r, g0, mass_norm, A_gamma, r_gamma):
    """
    Gamma-Sättigungsterm (physikalisch skaliert).
    Einheit: km²/s²/kpc, konsistent mit g_obs.
    Skaliert mit zentraler Beschleunigung g0 und normierter Masse.
    """
    return A_gamma * g0 * mass_norm * np.exp(-r / r_gamma)

def gamma_term_phase(r, g0, mass_norm, A_gamma,
                     r_gamma, r_max, f_lambda=1.0):
    """
    Gamma-Term mit Phasenwelle.
    Modelliert räumlich phasenversetzten GRB-Aktivitätsmuster.
    Wellenlänge skaliert mit Haloradius.
    """
    lambda_gamma = f_lambda * r_max
    envelope     = A_gamma * g0 * mass_norm * np.exp(-r / r_gamma)
    phase        = np.cos(2 * np.pi * r / lambda_gamma)
    return envelope * phase

print('Funktionen definiert.')

In [ ]:
# Normierte Masse für Gamma-Term
gal_M_max    = {gal: prof['M_bar'].max()
                for gal, prof in df_all.groupby('galaxy')}
M_max_global = max(gal_M_max.values())

gal_list = []

for gal, prof in df_all.groupby('galaxy'):
    prof   = prof.sort_values('r').copy()
    r_vals = prof['r'].values
    Vobs   = prof['Vobs'].values
    M_orig = prof['M_eff'].values

    r0        = r_vals[0]
    g0        = Vobs[0]**2 / r0
    r_max     = r_vals.max()
    mass_norm = gal_M_max[gal] / M_max_global
    V_max     = Vobs.max()

    # Original-Slopes mit verschiedenen n_inner
    slope_3 = inner_slope_from_profile(r_vals, M_orig, n_inner=3)
    slope_5 = inner_slope_from_profile(r_vals, M_orig, n_inner=5)
    slope_8 = inner_slope_from_profile(r_vals, M_orig, n_inner=8)

    row = {
        'galaxy':         gal,
        'r_max':          r_max,
        'M_bar_max':      gal_M_max[gal],
        'V_max':          V_max,
        'g0':             g0,
        'mass_norm':      mass_norm,
        'slope_3':        slope_3,
        'slope_5':        slope_5,
        'slope_8':        slope_8,
        'slope_original': slope_5,  # Standard
    }

    # Gamma-korrigierte Slopes
    for cfg in GAMMA_CONFIGS:
        if cfg['phase']:
            dg = gamma_term_phase(
                r_vals, g0, mass_norm,
                cfg['A_gamma'], cfg['r_gamma'],
                r_max, cfg.get('f_lambda', 1.0))
        else:
            dg = gamma_term(
                r_vals, g0, mass_norm,
                cfg['A_gamma'], cfg['r_gamma'])

        V_corr = np.sqrt(np.maximum(Vobs**2 - dg * r_vals, 0))
        M_corr = r_vals * V_corr**2
        sc     = inner_slope_from_profile(r_vals, M_corr)
        row[f"slope_{cfg['label']}"] = sc
        row[f"delta_{cfg['label']}"] = slope_5 - sc

    gal_list.append(row)

gal_df = pd.DataFrame(gal_list).dropna(subset=['slope_original'])

# Q-Regime mergen
gal_df = pd.merge(gal_df,
                  q_df[['galaxy','q','regime_q']],
                  on='galaxy', how='left')

# Größengruppen
gal_df['groesse'] = pd.cut(
    gal_df['r_max'],
    bins=[0, SCHWELLE_KLEIN, SCHWELLE_UEBERGANG, np.inf],
    labels=['Klein', 'Uebergang', 'Gross']
)

print(f'Galaxien total: {len(gal_df)}')
print(gal_df['groesse'].value_counts())

## 3. Größenabhängigkeit: drei Regime

In [ ]:
# Slope-Statistik pro Größengruppe
print('=== SLOPE STATISTIK PRO GRÖSSENGRUPPE ===')
for gruppe in ['Klein', 'Uebergang', 'Gross']:
    sub = gal_df[gal_df['groesse'] == gruppe]['slope_original'].dropna()
    print(f'\n{gruppe} (n={len(sub)}):')
    print(f'  Median={sub.median():.3f}, Mean={sub.mean():.3f}, Std={sub.std():.3f}')
    print(f'  Core-artig (<2.0): {(sub<2.0).mean()*100:.1f}%')
    print(f'  Cuspy (>2.5):      {(sub>2.5).mean()*100:.1f}%')

print('\n=== KORRELATION r_max vs slope ===')
for gruppe in ['Klein', 'Uebergang', 'Gross']:
    sub = gal_df[gal_df['groesse'] == gruppe][['r_max','slope_original']].dropna()
    if len(sub) > 5:
        r, p = stats.pearsonr(sub['r_max'], sub['slope_original'])
        print(f'{gruppe}: r={r:.3f}, p={p:.4f}, n={len(sub)}')

print('\n=== KRUSKAL-WALLIS TEST ===')
k_df = gal_df.dropna(subset=['slope_original','groesse'])
H, p = stats.kruskal(
    k_df[k_df['groesse']=='Klein']['slope_original'],
    k_df[k_df['groesse']=='Uebergang']['slope_original'],
    k_df[k_df['groesse']=='Gross']['slope_original']
)
print(f'H={H:.3f}, p={p:.6f}')

print('\n=== MANN-WHITNEY PAARWEISE ===')
paare = [('Klein','Uebergang'),('Uebergang','Gross'),('Klein','Gross')]
for g1, g2 in paare:
    s1 = k_df[k_df['groesse']==g1]['slope_original']
    s2 = k_df[k_df['groesse']==g2]['slope_original']
    _, p = stats.mannwhitneyu(s1, s2)
    print(f'{g1} vs {g2}: p={p:.6f}')

## 4. Gamma-Sättigungsterm: Klein vs. Groß

In [ ]:
delta_cols = [f"delta_{c['label']}" for c in GAMMA_CONFIGS]

def korr_rmse(df_sub, col):
    valid = df_sub[['r_max', col]].dropna()
    if len(valid) < 5:
        return np.nan, np.nan, np.nan
    r, p = stats.pearsonr(valid['r_max'], valid[col])
    sl, ic, _, _, _ = stats.linregress(valid['r_max'], valid[col])
    rmse = (valid[col] - (sl * valid['r_max'] + ic)).std()
    return r, p, rmse

slope_cols = ['slope_original'] + [f"slope_{c['label']}" for c in GAMMA_CONFIGS]

for gruppe in ['Klein', 'Uebergang', 'Gross']:
    sub = gal_df[gal_df['groesse'] == gruppe]
    print(f'\n=== {gruppe.upper()} (n={len(sub)}) ===')
    print(f'{"Modell":<24} {"r":>6} {"RMSE":>8} {"ΔRMSE":>10}')
    print('-' * 52)
    base_rmse = None
    for col in slope_cols:
        r, p, rmse = korr_rmse(sub, col)
        if base_rmse is None:
            base_rmse = rmse
            drmse = 'Baseline'
        else:
            drmse = f'{rmse-base_rmse:+.5f}'
        print(f'{col:<24} {r:>6.3f} {rmse:>8.4f} {drmse:>10}')

print('\n=== DELTA ALPHA MEDIAN PRO GRUPPE ===')
print(gal_df.groupby('groesse')[delta_cols].median().round(6))

print('\n=== ANTEIL NEGATIV (core-bildend) ===')
for col in delta_cols:
    by_g = gal_df.groupby('groesse').apply(
        lambda x: (x[col] < 0).mean()*100
    ).round(1)
    print(f'{col}: {by_g.to_dict()}')

## 5. Abbildungen für Paper

In [ ]:
# Abbildung 1: r_max vs. alpha mit drei Regimen und gleitendem Median

fig, ax = plt.subplots(figsize=(9, 6))

gruppe_colors = {'Klein': 'steelblue',
                 'Uebergang': 'darkorange',
                 'Gross': 'darkgreen'}

for gruppe, color in gruppe_colors.items():
    sub = gal_df[gal_df['groesse'] == gruppe]
    ax.scatter(sub['r_max'], sub['slope_original'],
               s=25, alpha=0.55, color=color,
               label=f"{gruppe} (n={len(sub)})")

# Gleitender Median
df_s  = gal_df.sort_values('r_max').dropna(subset=['slope_original'])
win   = 20
r_c, s_m, s_s = [], [], []
for i in range(len(df_s) - win):
    w = df_s.iloc[i:i+win]
    r_c.append(w['r_max'].median())
    s_m.append(w['slope_original'].median())
    s_s.append(w['slope_original'].std())

r_c, s_m, s_s = np.array(r_c), np.array(s_m), np.array(s_s)
ax.plot(r_c, s_m, 'k-', linewidth=2,
        label=f'Gleitender Median (n={win})')
ax.fill_between(r_c, s_m - s_s, s_m + s_s,
                alpha=0.12, color='black')

# Schwellen
ax.axvline(SCHWELLE_KLEIN, linestyle='--',
           color='steelblue', alpha=0.7,
           label=f'r={SCHWELLE_KLEIN} kpc')
ax.axvline(SCHWELLE_UEBERGANG, linestyle='--',
           color='darkgreen', alpha=0.7,
           label=f'r={SCHWELLE_UEBERGANG} kpc')

# Referenzlinien
ax.axhline(2.0, linestyle=':', color='navy',
           linewidth=1, alpha=0.6, label='NFW')
ax.axhspan(2.2, 2.8, alpha=0.08,
           color='steelblue', label='Obs. Core')

ax.set_xlabel('$r_\\mathrm{max}$ [kpc]', fontsize=12)
ax.set_ylabel(r'$\alpha = d\log M / d\log r$', fontsize=12)
ax.set_title(
    'Systemgröße als primärer Treiber der Core-Cusp Struktur\n'
    f'Gesamt: r={stats.pearsonr(gal_df["r_max"].dropna(), gal_df["slope_original"].dropna())[0]:.3f}, '
    f'p<0.0001',
    fontsize=11)
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.25)
ax.set_ylim(0.3, 4.2)

plt.tight_layout()
plt.savefig(OUT_PATH + 'fig_core_cusp_groesse.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Gespeichert: fig_core_cusp_groesse.png')

In [ ]:
# Abbildung 2: Slope-Verteilung pro Größengruppe

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, gruppe, color in zip(
    axes,
    ['Klein', 'Uebergang', 'Gross'],
    ['steelblue', 'darkorange', 'darkgreen']
):
    sub  = gal_df[gal_df['groesse'] == gruppe]['slope_original'].dropna()
    r_sub = gal_df[gal_df['groesse'] == gruppe]['r_max']

    ax.hist(sub, bins=15, color=color,
            alpha=0.75, edgecolor='white')
    ax.axvline(2.0, linestyle='--', color='navy',
               linewidth=1.5, label='NFW')
    ax.axvspan(2.2, 2.8, alpha=0.15,
               color='steelblue', label='Obs. Core')
    ax.axvline(sub.median(), linestyle='-',
               color='black', linewidth=2,
               label=f'Median={sub.median():.2f}')

    ax.set_xlabel(r'$\alpha$', fontsize=12)
    ax.set_ylabel('Anzahl', fontsize=11)
    ax.set_title(
        f'{gruppe}\n'
        f'n={len(sub)}, '
        f'$r_\\mathrm{{max}}$ median={r_sub.median():.1f} kpc\n'
        f'Core-artig: {(sub<2.0).mean()*100:.0f}%, '
        f'Cuspy: {(sub>2.5).mean()*100:.0f}%',
        fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle(
    'Verteilung innerer Steigungen: drei Regime der Core-Cusp Dynamik',
    fontsize=13)
plt.tight_layout()
plt.savefig(OUT_PATH + 'fig_core_cusp_verteilung.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Gespeichert: fig_core_cusp_verteilung.png')

In [ ]:
# Abbildung 3: Gamma-Term RMSE-Vergleich Klein vs. Groß

fig, ax = plt.subplots(figsize=(9, 5))

labels  = [c['label'] for c in GAMMA_CONFIGS]
rmse_k, rmse_u, rmse_g = [], [], []

for cfg in GAMMA_CONFIGS:
    col = f"slope_{cfg['label']}"
    for lst, gruppe in [(rmse_k,'Klein'),(rmse_u,'Uebergang'),(rmse_g,'Gross')]:
        _, _, r = korr_rmse(gal_df[gal_df['groesse']==gruppe], col)
        lst.append(r)

_, _, base_k = korr_rmse(gal_df[gal_df['groesse']=='Klein'],     'slope_original')
_, _, base_u = korr_rmse(gal_df[gal_df['groesse']=='Uebergang'], 'slope_original')
_, _, base_g = korr_rmse(gal_df[gal_df['groesse']=='Gross'],     'slope_original')

x = np.arange(len(labels))
w = 0.28

ax.bar(x - w, [r - base_k for r in rmse_k], w,
       label=f'Klein (r<{SCHWELLE_KLEIN} kpc)',
       color='steelblue', alpha=0.8)
ax.bar(x,     [r - base_u for r in rmse_u], w,
       label=f'Übergang ({SCHWELLE_KLEIN}–{SCHWELLE_UEBERGANG} kpc)',
       color='darkorange', alpha=0.8)
ax.bar(x + w, [r - base_g for r in rmse_g], w,
       label=f'Groß (r≥{SCHWELLE_UEBERGANG} kpc)',
       color='darkgreen', alpha=0.8)

ax.axhline(0, linestyle='--', color='black',
           linewidth=1.5, label='Baseline')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel('ΔRMSE (Gamma − Original)')
ax.set_title(
    'Gamma-Term: RMSE-Verbesserung pro Größengruppe\n'
    'Negativ = Verbesserung gegenüber Baseline')
ax.legend(fontsize=9)
ax.grid(alpha=0.25, axis='y')
plt.tight_layout()
plt.savefig(OUT_PATH + 'fig_gamma_rmse_vergleich.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Gespeichert: fig_gamma_rmse_vergleich.png')

In [ ]:
# Abbildung 4: Gamma-Sättigungsprofil Peak vs. Diffus

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

r_test = np.linspace(0.1, 50, 500)

for ax, regime, color in zip(
    axes, ['Peak', 'Diffus'], ['steelblue', 'green']
):
    sub = gal_df[gal_df['regime_q'] == regime]
    if len(sub) == 0:
        continue

    med_g0        = sub['g0'].median()
    med_mass_norm = sub['mass_norm'].median()
    med_r_max     = sub['r_max'].median()

    for cfg in GAMMA_CONFIGS[:3]:  # nur einfache Terme
        dg = gamma_term(r_test, med_g0, med_mass_norm,
                        cfg['A_gamma'], cfg['r_gamma'])
        ls = {'g_schwach':'--','g_mittel':'-','g_stark':'-.'}[cfg['label']]
        ax.plot(r_test, dg, linestyle=ls,
                color=color, alpha=0.8,
                label=f"{cfg['label']} (A={cfg['A_gamma']})")

    ax.axhline(0, linestyle=':', color='gray', linewidth=0.8)
    ax.axvline(med_r_max, linestyle=':',
               color='black', alpha=0.4,
               label=f'r_max={med_r_max:.1f} kpc')
    ax.fill_between(
        r_test,
        0.15 * med_g0 * med_mass_norm * np.exp(-r_test / 2.0),
        0, alpha=0.08, color=color, label='Einhüllende (mittel)')

    ax.set_xlabel('r [kpc]', fontsize=12)
    ax.set_ylabel(r'$\Delta g_\gamma$ [km²/s²/kpc]', fontsize=11)
    ax.set_title(
        f'{regime}-System\n'
        f'(median $r_\\mathrm{{max}}$={med_r_max:.1f} kpc, '
        f'$g_0$={med_g0:.0f} km²/s²/kpc)',
        fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    ax.set_xlim(0, min(50, med_r_max * 2))

plt.suptitle(
    'Gamma-Sättigungsprofil: Peak vs. Diffus\n'
    'Diffuse Systeme zeigen stärkere absolute Feldanregung',
    fontsize=12)
plt.tight_layout()
plt.savefig(OUT_PATH + 'fig_gamma_profil_regime.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Gespeichert: fig_gamma_profil_regime.png')

## 6. Ergebnisse speichern

In [ ]:
# Hauptergebnisse
gal_df.to_csv(OUT_PATH + 'core_cusp_uqsh_results.csv', index=False)

# Zusammenfassung pro Gruppe
summary_cols = ['slope_3','slope_5','slope_8','slope_original',
                'r_max','V_max','mass_norm']
summary = gal_df.groupby('groesse')[summary_cols].agg(
    ['median','std','count']
).round(3)
summary.to_csv(OUT_PATH + 'core_cusp_summary.csv')

print('Gespeichert:')
print('  core_cusp_uqsh_results.csv   (alle Galaxien)')
print('  core_cusp_summary.csv        (Zusammenfassung pro Gruppe)')
print('  fig_core_cusp_groesse.png')
print('  fig_core_cusp_verteilung.png')
print('  fig_gamma_rmse_vergleich.png')
print('  fig_gamma_profil_regime.png')
print()
print('Analyse abgeschlossen.')